In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes trl

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-v0.1"

# ✅ 4-bit config (REQUIRED)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,   # ✅ instead of load_in_4bit
    device_map="auto",
    torch_dtype=torch.float16,
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [3]:
tokenizer.pad_token = tokenizer.eos_token

In [4]:
print(model.dtype)

torch.float16


In [5]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # works well for Mistral
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 7,248,547,840 || trainable%: 0.0940


In [6]:
from datasets import Dataset
import json

with open("/kaggle/input/datasets/kmldas/indiclegalqa-dataset/IndicLegalQA Dataset_10K.json") as f:
    data = json.load(f)

def format_data(example):
    date = (
        example.get("judgment_date") or 
        example.get("judgement_date") or 
        "Unknown"
    )

    return {
        "text": f"""### Case Name: {example.get('case_name', '')}
### Date: {date}
### Question: {example.get('question', '')}
### Answer: {example.get('answer', '')}"""
    }

dataset = Dataset.from_list([format_data(x) for x in data])

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments

tokenizer.model_max_length = 512

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=10,

    fp16=False,   # ❌ TURN OFF
    bf16=False,   # ❌ TURN OFF

    optim="adamw_torch",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
)



Adding EOS to train dataset:   0%|          | 0/10002 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10002 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10002 [00:00<?, ? examples/s]

In [8]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.is_bf16_supported())

Tesla T4
True


In [ ]:
print(training_args)
print(model.dtype)

In [ ]:
import torch
torch.backends.cuda.matmul.allow_bf16 = False

In [9]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.621905
20,1.321927
30,1.291574
40,1.207295
50,1.236345
60,1.216243
70,1.278683
80,1.238168
90,1.246307
100,1.186668


TrainOutput(global_step=5001, training_loss=1.0694883826302901, metrics={'train_runtime': 4605.7718, 'train_samples_per_second': 2.172, 'train_steps_per_second': 1.086, 'total_flos': 5.63460668854272e+16, 'train_loss': 1.0694883826302901})

In [15]:
# Create the specific folder for your "Shadow" adapter
import os
save_dir = "/kaggle/working/.virtual_documents/mistral_lora_adapter"
os.makedirs(save_dir, exist_ok=True)

# Save the delta weights (LoRA) and the tokenizer
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"✅ Adapter saved to {save_dir}. Now perform a Quick Save.")

✅ Adapter saved to /kaggle/working/.virtual_documents/mistral_lora_adapter. Now perform a Quick Save.


In [13]:
import os

# Define the local save path
adapter_path = "/kaggle/working/mistral_legal_adapter"

# Save the adapter (the learned LoRA weights)
model.save_pretrained(adapter_path)

# Save the tokenizer (essential for matching the model's vocabulary)
tokenizer.save_pretrained(adapter_path)

print(f"✅ Adapter saved locally to: {adapter_path}")

✅ Adapter saved locally to: /kaggle/working/mistral_legal_adapter


In [16]:
import shutil
shutil.make_archive("mistral_legal_adapter", 'zip', "/kaggle/working/mistral_legal_adapter")
print("✅ Zip file created. You can download 'mistral_legal_adapter.zip' from the Output sidebar.")

✅ Zip file created. You can download 'mistral_legal_adapter.zip' from the Output sidebar.
